In [1]:
import numpy as np
import tensorflow as tf

In [2]:
from tensorflow.keras import models,layers,datasets,preprocessing

In [3]:
tf.random.set_seed(42)
np.random.seed(42)

In [4]:
# Hyperparameters
VOCAB_SIZE = 10000   # Max unique words to keep
MAX_LEN = 200        # Fixed sequence length (words per review)
EMBED_DIM = 64       # Word vector dimension
GRU_UNITS = 64       # Hidden units per GRU direction (64 fwd + 64 bwd = 128 total)
NUM_HEADS = 4        # Parallel attention heads
KEY_DIM = 16         # Query/Key projection dimension
BATCH_SIZE = 64
EPOCHS = 5

In [5]:
print("loading IMDB movie review dataset")
(x_train,y_train),(x_test,y_test)=datasets.imdb.load_data(num_words=VOCAB_SIZE)

loading IMDB movie review dataset


In [7]:
x_train=preprocessing.sequence.pad_sequences(x_train,maxlen=MAX_LEN,padding='post', truncating='post')
x_test=preprocessing.sequence.pad_sequences(x_test,maxlen=MAX_LEN,padding='post', truncating='post')

In [8]:
print(f"Train shapes: x={x_train.shape}, y={y_train.shape}")
print(f"Test shapes:  x={x_test.shape}, y={y_test.shape}")

Train shapes: x=(25000, 200), y=(25000,)
Test shapes:  x=(25000, 200), y=(25000,)


In [9]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding, GRU, Bidirectional, MultiHeadAttention, 
    LayerNormalization, GlobalAveragePooling1D, Dense, Dropout, Layer
)

In [10]:
model = Sequential()
model.add(Embedding(input_dim=10000, output_dim=64, input_length=200))
model.add(Bidirectional(GRU(64, dropout=0.2))) # Returns single context vector automatically
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))

model.summary()

c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Bidirectional, GRU, Dense, Dropout

model = Sequential([
    Input(shape=(200,)),  # Defines sequence length (e.g., 200 tokens)
    Embedding(input_dim=10000, output_dim=128),
    Bidirectional(GRU(64)),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # Or Dense(46, activation='softmax') for multi-class
])

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,362,817 (5.20 MB)

 Trainable params: 1,362,817 (5.20 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [15]:
history = model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=5,
    validation_split=0.2,
    verbose=1
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 47s 134ms/step - accuracy: 0.7194 - loss: 0.5256 - val_accuracy: 0.8362 - val_loss: 0.4088
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 131ms/step - accuracy: 0.8755 - loss: 0.3162 - val_accuracy: 0.8420 - val_loss: 0.3811
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 157ms/step - accuracy: 0.9060 - loss: 0.2470 - val_accuracy: 0.8214 - val_loss: 0.4291
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 49s 157ms/step - accuracy: 0.9281 - loss: 0.1920 - val_accuracy: 0.8482 - val_loss: 0.4315
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 121s 129ms/step - accuracy: 0.9549 - loss: 0.1279 - val_accuracy: 0.8504 - val_loss: 0.5062


In [16]:
# 6. Evaluate on Test Data
test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=BATCH_SIZE, verbose=1)
print(f"\n✅ Test Accuracy: {test_acc * 100:.2f}%")
print(f"📊 Test Loss:     {test_loss:.4f}")

391/391 ━━━━━━━━━━━━━━━━━━━━ 27s 69ms/step - accuracy: 0.8351 - loss: 0.5775

✅ Test Accuracy: 83.51%
📊 Test Loss:     0.5775


In [17]:
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

In [20]:
model=Sequential([
    Input(shape=(200,)),
    Embedding(input_dim=10000,output_dim=128),
    Bidirectional(GRU(64,dropout=0.3,recurrent_dropout=0.2)),
    Dense(32, activation='relu',kernel_regularizer=l2(0.001)),
    Dropout(0.5),
    Dense(1, activation='sigmoid',kernel_regularizer=l2(0.001))  # Or Dense(46, activation='softmax') for multi-class
])

In [21]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [22]:
# 2. Define Callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=2, 
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=1, 
        verbose=1
    )
]

In [23]:
# 3. Train Model
history = model.fit(
    x_train, y_train,
    batch_size=64,
    epochs=10,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 103s 314ms/step - accuracy: 0.6322 - loss: 0.6407 - val_accuracy: 0.7780 - val_loss: 0.4820 - learning_rate: 0.0010
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 329ms/step - accuracy: 0.8014 - loss: 0.4751
Epoch 2: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
313/313 ━━━━━━━━━━━━━━━━━━━━ 106s 339ms/step - accuracy: 0.8188 - loss: 0.4450 - val_accuracy: 0.7816 - val_loss: 0.5159 - learning_rate: 0.0010
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 109s 348ms/step - accuracy: 0.8724 - loss: 0.3505 - val_accuracy: 0.8170 - val_loss: 0.4434 - learning_rate: 5.0000e-04
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 322ms/step - accuracy: 0.8805 - loss: 0.3247
Epoch 4: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
313/313 ━━━━━━━━━━━━━━━━━━━━ 104s 333ms/step - accuracy: 0.8946 - loss: 0.2958 - val_accuracy: 0.8214 - val_loss: 0.4444 - learning_rate: 5.0000e-04
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 0s 312ms/step - accura

In [24]:
#4. Evaluate Test Set
test_loss, test_acc = model.evaluate(x_test, y_test, batch_size=64)
print(f"\n✅ Test Accuracy: {test_acc * 100:.2f}%")

391/391 ━━━━━━━━━━━━━━━━━━━━ 14s 36ms/step - accuracy: 0.8067 - loss: 0.4584

✅ Test Accuracy: 80.67%


In [25]:
import numpy as np
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences

# 1. Load IMDB word index
word_index = imdb.get_word_index()

# 2. Function to preprocess and classify custom text
def predict_sentiment(review_text, max_len=200, vocab_size=10000):
    # Clean and tokenize input text
    words = review_text.lower().replace(".", "").replace(",", "").replace("!", "").split()
    
    # Map words to indices (IMDB uses +3 index offset)
    encoded = [1]  # Token 1 = Start sequence
    for word in words:
        if word in word_index and (word_index[word] + 3) < vocab_size:
            encoded.append(word_index[word] + 3)
        else:
            encoded.append(2)  # Token 2 = Out-of-Vocabulary (OOV)
            
    # Pad sequence to match training shape (200 tokens)
    padded = pad_sequences([encoded], maxlen=max_len, padding='post', truncating='post')
    
    # Generate prediction probability
    prob = model.predict(padded, verbose=0)[0][0]
    
    # Interpret output
    sentiment = "POSITIVE 😃" if prob >= 0.5 else "NEGATIVE 🙁"
    confidence = prob if prob >= 0.5 else (1 - prob)
    
    print(f"Review: \"{review_text}\"")
    print(f"Prediction: {sentiment} | Score: {prob:.4f} | Confidence: {confidence * 100:.2f}%\n")

# 3. Test on custom reviews
print("--- MODEL INFERENCE CHECKS ---\n")

predict_sentiment("An absolute masterpiece! The acting was incredible and the plot kept me engaged from start to finish.")

predict_sentiment("Terrible movie. The pacing was completely off, dialogue felt forced, and I was bored within 20 minutes.")

predict_sentiment("It had some cool visual effects, but overall the story was predictable and weak.")

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 1s 1us/step
--- MODEL INFERENCE CHECKS ---

Review: "An absolute masterpiece! The acting was incredible and the plot kept me engaged from start to finish."
Prediction: POSITIVE 😃 | Score: 0.9637 | Confidence: 96.37%

Review: "Terrible movie. The pacing was completely off, dialogue felt forced, and I was bored within 20 minutes."
Prediction: NEGATIVE 🙁 | Score: 0.0331 | Confidence: 96.69%

Review: "It had some cool visual effects, but overall the story was predictable and weak."
Prediction: POSITIVE 😃 | Score: 0.9200 | Confidence: 92.00%



In [29]:
predict_sentiment('it was a wonderfull experience,could relate to it')

Review: "it was a wonderfull experience,could relate to it"
Prediction: POSITIVE 😃 | Score: 0.9653 | Confidence: 96.53%



In [28]:
predict_sentiment('pretty bad,')

Review: "pretty bad,"
Prediction: POSITIVE 😃 | Score: 0.8841 | Confidence: 88.41%



false positives caused by a classic recurrent neural network flaw: padding='post'.

When a 5-word review like "pretty bad" is post-padded to 200 tokens, the GRU reads the actual words in the first 5 timesteps and then processes 195 consecutive zero tokens. By timestep 200, the GRU's hidden memory state has decayed, washing out the sentiment and defaulting to the dataset's baseline positive class bias.

Key Technical Causes
Post-Padding Memory Decay: With padding='post', the final hidden state passed to the Dense layer is dominated by zero-token padding rather than the actual words.

Keyword Bias in Short Input: In the IMDB training corpus, words like "pretty" (e.g., "pretty good") and "cool" appear heavily in positive contexts. Without strong final-timestep sequence context, the model over-indexes on these positive words.

Index Offset Mismatch: If punctuation or unknown word mappings fall back to Out-Of-Vocabulary (OOV token 2), the model loses critical negative modifiers like "weak" or "bad".